# 08 - Model Evaluation

## Objective

This notebook evaluates the candidate final model selected in notebook 07 using the chronological validation window and the untouched holdout test set.

The evaluation includes probability calibration, threshold tuning, lift analysis, subgroup error analysis, final model retraining, holdout metrics, and saved model artifacts.

#### Load project configuration and modeling checkpoints

In [0]:
# Load the project configuration and modeling checkpoints

from __future__ import annotations

import json

from config import project_config as cfg
from pyspark.sql import functions as F
from utils import model_evaluation as meval
from utils.model_training import evaluate_tuning_predictions


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the model-training notebook (07) before continuing."
        )


def require_file(file_path: str) -> None:
    try:
        dbutils.fs.head(file_path, 1)
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found. "
            "Run the model-training notebook (07) before continuing."
        ) from exc


require_file(cfg.MODEL_FEATURE_MANIFEST_PATH)
require_file(cfg.CANDIDATE_SELECTION_PATH)

feature_manifest = json.loads(
    dbutils.fs.head(cfg.MODEL_FEATURE_MANIFEST_PATH, 1000000)
)
candidate_selection = json.loads(
    dbutils.fs.head(cfg.CANDIDATE_SELECTION_PATH, 1000000)
)

TARGET_COLUMN = feature_manifest["target_column"]
CATEGORICAL_COLUMNS = feature_manifest["categorical_columns"]
NUMERICAL_COLUMNS = feature_manifest["numerical_columns"]
MODEL_INPUT_COLUMNS = feature_manifest["model_input_columns"]
HASH_VECTOR_SIZE = feature_manifest["hash_vector_size"]
SELECTED_MODEL_NAME = candidate_selection["selected_model_name"]
SELECTED_MODEL_PARAMETERS = candidate_selection["selected_model_parameters"]

for table_name in [
    cfg.MODELING_TRAIN_HASHED_TABLE,
    cfg.MODELING_VALIDATION_HASHED_TABLE,
    cfg.MODELING_TEST_HASHED_TABLE,
    cfg.MODELING_TRAIN_HIST_TABLE,
    cfg.MODELING_VALIDATION_HIST_TABLE,
    cfg.MODELING_TEST_HIST_TABLE,
    cfg.TUNED_MODEL_COMPARISON_TABLE,
]:
    require_table(table_name)

df_train_hashed = spark.table(cfg.MODELING_TRAIN_HASHED_TABLE)
df_validation_hashed = spark.table(cfg.MODELING_VALIDATION_HASHED_TABLE)
df_test_hashed = spark.table(cfg.MODELING_TEST_HASHED_TABLE)
df_train_hist = spark.table(cfg.MODELING_TRAIN_HIST_TABLE)
df_validation_hist = spark.table(cfg.MODELING_VALIDATION_HIST_TABLE)
df_test_hist = spark.table(cfg.MODELING_TEST_HIST_TABLE)
tuned_model_comparison = spark.table(cfg.TUNED_MODEL_COMPARISON_TABLE)
selected_model_summary = (
    tuned_model_comparison.filter(F.col("MODEL") == SELECTED_MODEL_NAME)
)

print("Modeling checkpoints loaded successfully.")
print(f"Selected model: {SELECTED_MODEL_NAME}")
print(f"Selected parameters: {SELECTED_MODEL_PARAMETERS}")


#### Validation-period calibration, threshold tuning, and error analysis

Before retraining on the combined training and validation periods, the selected Logistic Regression configuration is evaluated on the untouched September–October validation window. The model is trained only on the January–August training period to avoid leakage.

This section adds the probability-calibration, threshold-selection, lift, and subgroup analyses recommended for the final modeling stage.

In [0]:
from pyspark.ml.classification import LogisticRegression
from utils import model_evaluation as meval

candidate_lr_estimator = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    regParam=SELECTED_MODEL_PARAMETERS["regParam"],
    elasticNetParam=SELECTED_MODEL_PARAMETERS["elasticNetParam"],
    maxIter=cfg.SELECTED_LR_MAX_ITER,
    standardization=True,
    family="binomial",
)

candidate_lr_model = candidate_lr_estimator.fit(df_train_hashed)
validation_predictions = candidate_lr_model.transform(df_validation_hashed)

VALIDATION_PERIOD_METRICS = evaluate_tuning_predictions(validation_predictions, TARGET_COLUMN)

display(
    spark.createDataFrame(
        [
            {
                "MODEL": cfg.SELECTED_MODEL_NAME,
                "DATASET": "Chronological Validation (Sep–Oct 2025)",
                **{
                    metric_name: round(metric_value, 4)
                    for metric_name, metric_value in VALIDATION_PERIOD_METRICS.items()
                },
            }
        ]
    )
)


In [0]:
import matplotlib.pyplot as plt

validation_probability_frame = (
    validation_predictions
    .select(
        F.col(TARGET_COLUMN).alias("label"),
        F.element_at(F.vector_to_array("probability"), 2).alias("delay_probability"),
    )
    .sample(
        withReplacement=False,
        fraction=cfg.CALIBRATION_SAMPLE_FRACTION,
        seed=cfg.RANDOM_SEED,
    )
    .toPandas()
)

VALIDATION_BRIER_SCORE = meval.brier_score(
    validation_probability_frame["label"],
    validation_probability_frame["delay_probability"],
)

calibration_summary = meval.calibration_summary(
    validation_probability_frame["label"],
    validation_probability_frame["delay_probability"],
    n_bins=cfg.CALIBRATION_BINS,
)

display(calibration_summary)
print(f"Validation Brier score: {VALIDATION_BRIER_SCORE:.4f}")

calibration_figure = meval.plot_calibration_curve(
    validation_probability_frame["label"],
    validation_probability_frame["delay_probability"],
    n_bins=cfg.CALIBRATION_BINS,
)
display(calibration_figure)
plt.close(calibration_figure)


In [0]:
threshold_results = meval.threshold_search(
    validation_probability_frame["label"],
    validation_probability_frame["delay_probability"],
    threshold_min=cfg.THRESHOLD_SEARCH_MIN,
    threshold_max=cfg.THRESHOLD_SEARCH_MAX,
    threshold_step=cfg.THRESHOLD_SEARCH_STEP,
)

display(threshold_results)

OPTIMAL_DECISION_THRESHOLD = float(
    threshold_results.sort_values("delay_f1", ascending=False).iloc[0]["threshold"]
)

lift_results = meval.lift_at_top_percentiles(
    validation_probability_frame["label"],
    validation_probability_frame["delay_probability"],
    percentiles=cfg.TOP_RISK_PERCENTILES,
)

display(lift_results)
print(f"Selected validation threshold: {OPTIMAL_DECISION_THRESHOLD:.2f}")


In [0]:
validation_scored = (
    df_validation_hist.select(
        *cfg.BUSINESS_KEY_COLUMNS,
        *cfg.SUBGROUP_ERROR_COLUMNS,
        F.col(TARGET_COLUMN).alias("label"),
    )
    .join(
        validation_predictions.select(
            *cfg.BUSINESS_KEY_COLUMNS,
            F.element_at(F.vector_to_array("probability"), 2).alias("delay_probability"),
        ),
        on=cfg.BUSINESS_KEY_COLUMNS,
        how="inner",
    )
)

subgroup_frame = validation_scored.toPandas()
subgroup_results = meval.subgroup_error_analysis(
    subgroup_frame,
    group_columns=cfg.SUBGROUP_ERROR_COLUMNS,
    label_column="label",
    probability_column="delay_probability",
    threshold=OPTIMAL_DECISION_THRESHOLD,
    min_group_size=cfg.MIN_ROUTE_FLIGHTS,
)

display(subgroup_results)


#### Save the selected model and supporting artifacts

The selected Logistic Regression configuration is retrained before being saved as a Spark ML model.

The final training dataset combines the original training and validation periods. This allows the selected model to learn from all historical observations available before the test period while preserving the test dataset for final holdout evaluation.

The saved artifacts will include:

- The fitted Spark ML Logistic Regression model
- The selected hyperparameters
- Model evaluation metrics
- Dataset and feature metadata

Saving these artifacts supports reproducibility and allows the model to be loaded later for batch inference, dashboard integration, and explainability analysis.


#### Train the Final Selected Model Using the Tuning Sampling Strategy

The selected Logistic Regression configuration is trained using the combined training and validation periods while preserving the same stratified sampling strategy used during chronological hyperparameter tuning.

The on-time class is sampled at 10%, while the delayed-flight class is sampled at 30%. This maintains consistency between model selection and final training and reduces the tendency of the model to predict nearly all observations as on-time.

The untouched test dataset remains unsampled and is used only for final holdout evaluation.


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.sql import functions as F


# Combine the chronological training and validation periods.
df_final_training_full = (
    df_train_hashed
    .unionByName(df_validation_hashed)
)


# Apply the same class-specific sampling strategy used during tuning.
df_final_training_sampled = (
    df_final_training_full
    .sampleBy(
        col=TARGET_COLUMN,
        fractions=cfg.TREE_TUNING_SAMPLE_FRACTIONS,
        seed=cfg.RANDOM_SEED,
    )
)


# Validate the sampled final-training distribution.
print("Sampled final-training class distribution:")

df_final_training_sampled.groupBy(
    TARGET_COLUMN
).count().orderBy(
    TARGET_COLUMN
).show()


# Create the selected Logistic Regression estimator.
final_logistic_regression_estimator = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    regParam=SELECTED_MODEL_PARAMETERS["regParam"],
    elasticNetParam=SELECTED_MODEL_PARAMETERS["elasticNetParam"],
    maxIter=cfg.SELECTED_LR_MAX_ITER,
    standardization=True,
    family="binomial",
)


# Train the final selected model.
final_selected_model = final_logistic_regression_estimator.fit(
    df_final_training_sampled
)


print("Final sampled Logistic Regression model trained successfully.")
print("Selected regParam: 0.001")
print("Selected elasticNetParam: 0.0")

#### Holdout test evaluation

The retrained Logistic Regression model is evaluated using the untouched November–December 2025 holdout test dataset.

Unlike the chronological validation folds used during hyperparameter tuning, the holdout dataset was never used during model development, parameter selection, or final model training. Consequently, this evaluation provides the most realistic estimate of how the selected model is expected to perform on future unseen flight records.

The same evaluation metrics used throughout the project are calculated to support direct comparison between the tuning results and the final deployment-ready model.


In [0]:
final_test_predictions = final_selected_model.transform(
    df_test_hashed
)

FINAL_TEST_METRICS = evaluate_tuning_predictions(final_test_predictions, TARGET_COLUMN)

display(
    spark.createDataFrame(
        [
            {
                "MODEL": "Logistic Regression",
                "DATASET": "Holdout Test",
                **{
                    metric_name: round(metric_value, 4)
                    for metric_name, metric_value
                    in FINAL_TEST_METRICS.items()
                },
            }
        ]
    )
)

#### Holdout confusion matrix and ranking curves

The holdout confusion matrix and ROC/PR curves are generated from the saved test predictions using the dynamically computed `FINAL_TEST_METRICS` table above.

In [0]:
import matplotlib.pyplot as plt


holdout_evaluation_frame = (
    final_test_predictions
    .select(
        F.col(TARGET_COLUMN).alias("label"),
        F.col("prediction").alias("prediction"),
        F.element_at(F.vector_to_array("probability"), 2).alias(
            "delay_probability"
        ),
    )
    .sample(
        withReplacement=False,
        fraction=cfg.CALIBRATION_SAMPLE_FRACTION,
        seed=cfg.RANDOM_SEED,
    )
    .toPandas()
)

holdout_confusion_summary = meval.confusion_matrix_summary(
    holdout_evaluation_frame["label"],
    holdout_evaluation_frame["prediction"],
)

display(holdout_confusion_summary)

confusion_figure = meval.plot_confusion_matrix(
    holdout_evaluation_frame["label"],
    holdout_evaluation_frame["prediction"],
)
display(confusion_figure)
plt.close(confusion_figure)


In [0]:
import matplotlib.pyplot as plt


roc_pr_figure = meval.plot_roc_pr_curves(
    holdout_evaluation_frame["label"],
    holdout_evaluation_frame["delay_probability"],
)
display(roc_pr_figure)
plt.close(roc_pr_figure)


#### Holdout calibration diagnostics

The holdout test set is evaluated once for calibration diagnostics. No threshold tuning or model changes are performed using the test window.


In [0]:
test_probability_frame = (
    final_test_predictions
    .select(
        F.col(TARGET_COLUMN).alias("label"),
        F.element_at(F.vector_to_array("probability"), 2).alias("delay_probability"),
    )
    .sample(
        withReplacement=False,
        fraction=cfg.CALIBRATION_SAMPLE_FRACTION,
        seed=cfg.RANDOM_SEED,
    )
    .toPandas()
)

TEST_BRIER_SCORE = meval.brier_score(
    test_probability_frame["label"],
    test_probability_frame["delay_probability"],
)

test_calibration_summary = meval.calibration_summary(
    test_probability_frame["label"],
    test_probability_frame["delay_probability"],
    n_bins=cfg.CALIBRATION_BINS,
)

display(test_calibration_summary)
print(f"Holdout Brier score: {TEST_BRIER_SCORE:.4f}")

test_calibration_figure = meval.plot_calibration_curve(
    test_probability_frame["label"],
    test_probability_frame["delay_probability"],
    n_bins=cfg.CALIBRATION_BINS,
)
display(test_calibration_figure)
plt.close(test_calibration_figure)


#### Holdout test interpretation

The holdout metrics displayed above are generated dynamically from `FINAL_TEST_METRICS` and should be used as the official test-set results in the final report.

Compared with the chronological validation results obtained during hyperparameter tuning, the holdout evaluation typically shows lower delayed-flight recall and delayed-flight F1-score. This reduction is expected because the holdout dataset contains completely unseen observations and therefore provides a more realistic estimate of operational performance.

The selected Logistic Regression model remains the candidate final model based on tuning-fold comparison. Random Forest and Gradient-Boosted Trees were not re-evaluated on the holdout test set.

#### Save the Trained Spark ML Model

The retrained Logistic Regression model is saved as a native Spark ML model.

Saving the model enables future loading for batch inference, operational deployment, dashboard integration, and explainability analysis without requiring retraining.

The model is stored in the Databricks workspace using Spark ML's native persistence format.


In [0]:
from pathlib import Path

MODEL_SAVE_PATH = cfg.SELECTED_MODEL_PATH

final_selected_model.write().overwrite().save(
    MODEL_SAVE_PATH
)

print("Final model saved successfully.")
print(f"Location: {MODEL_SAVE_PATH}")

#### Save Model Metadata

The selected hyperparameters and evaluation metrics are saved to support reproducibility and future model maintenance.

These metadata describe the final production model without requiring the tuning process to be rerun.


In [0]:
import json

selected_metrics_row = (
    selected_model_summary
    .select(
        "MODEL",
        "PARAMETERS",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
        "DELAY_PRECISION",
        "DELAY_RECALL",
        "DELAY_F1",
        "TRAINING_SECONDS",
    )
    .first()
)

if selected_metrics_row is None:
    raise ValueError(
        "Selected model metadata could not be retrieved."
    )

MODEL_METADATA = {
    "model": selected_metrics_row["MODEL"],
    "target": TARGET_COLUMN,
    "parameters": SELECTED_MODEL_PARAMETERS,
    "parameter_summary": selected_metrics_row["PARAMETERS"],
    "hash_vector_size": HASH_VECTOR_SIZE,
    "categorical_features": CATEGORICAL_COLUMNS,
    "numerical_features": NUMERICAL_COLUMNS,
    "total_raw_predictors": len(MODEL_INPUT_COLUMNS),
    "selected_after_hyperparameter_tuning": True,
    "final_training_period": "January 2025 to October 2025",
    "holdout_test_period": "November 2025 to December 2025",
    "model_format": "Spark ML",
}

dbutils.fs.put(
    (
        cfg.SELECTED_MODEL_METADATA_PATH
    ),
    json.dumps(MODEL_METADATA, indent=4),
    overwrite=True,
)

print("Dynamic model metadata saved successfully.")

#### Save Evaluation Metrics

The final evaluation metrics of the selected model are saved to provide a permanent record of the model's predictive performance.

These metrics support future model monitoring and performance comparison.


In [0]:
MODEL_METRICS = {
    "accuracy": float(selected_metrics_row["ACCURACY"]),
    "precision": float(selected_metrics_row["PRECISION"]),
    "recall": float(selected_metrics_row["RECALL"]),
    "f1_score": float(selected_metrics_row["F1_SCORE"]),
    "roc_auc": float(selected_metrics_row["ROC_AUC"]),
    "pr_auc": float(selected_metrics_row["PR_AUC"]),
    "delay_precision": float(
        selected_metrics_row["DELAY_PRECISION"]
    ),
    "delay_recall": float(
        selected_metrics_row["DELAY_RECALL"]
    ),
    "delay_f1": float(
        selected_metrics_row["DELAY_F1"]
    ),
    "average_training_seconds": float(
        selected_metrics_row["TRAINING_SECONDS"]
    ),
    "metric_source": (
        "Average performance across four chronological "
        "hyperparameter-tuning validation folds"
    ),
    "validation_brier_score": float(VALIDATION_BRIER_SCORE),
    "holdout_brier_score": float(TEST_BRIER_SCORE),
    "selected_validation_threshold": float(OPTIMAL_DECISION_THRESHOLD),
    "holdout_metrics": {
        metric_name: float(metric_value)
        for metric_name, metric_value in FINAL_TEST_METRICS.items()
    },
}

dbutils.fs.put(
    (
        cfg.SELECTED_MODEL_METRICS_PATH
    ),
    json.dumps(MODEL_METRICS, indent=4),
    overwrite=True,
)

print("Dynamic model evaluation metrics saved successfully.")